In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 

In [ ]:
metadata = pd.read_csv('HAM10000_metadata.csv')

In [ ]:
metadata.sample(5)

In [ ]:
metadata['dx'].value_counts()

In [ ]:
plt.figure(figsize=(10,6))
metadata['dx'].value_counts().plot(kind='bar')
plt.title("Lesion Types")
plt.ylabel("Frequency")
plt.show()

In [ ]:
metadata.iloc[1]

In [ ]:
image_id1 = metadata.iloc[1]['image_id']

In [ ]:
image_id1

In [ ]:
import os 
from PIL import Image

In [ ]:
image_id1_path = os.path.join('HAM10000_images_part_1/', f'{image_id1}.jpg')

In [ ]:
image_1 = Image.open(image_id1_path)

In [ ]:
image_1

In [ ]:
import torchvision.transforms.v2 as T

In [ ]:
tranform = T.Compose([
    T.Resize((224,224)), 
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
tranform(image_1)

In [ ]:
metadata['dx'].value_counts()

In [ ]:
label = {'nv' : 0, 'mel' : 1, 'bkl' : 2, 'bcc' : 3, 'akiec' : 4, 'vasc' : 5, 'df' : 6}

In [ ]:
metadata['dx'] = metadata['dx'].map(label)

In [ ]:
import torch

In [ ]:
class CustomDataSetLoader(torch.utils.data.Dataset): 
    def __init__(self, metadata, img_dir, transform = None, target_transform = None): 
        self.metadata = metadata 
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self): 
        return len(self.metadata)

    def __getitem__(self, idx): 
        img_id = self.metadata.iloc[idx]['image_id']
        if os.path.exists(os.path.join(self.img_dir + "_1", img_id + ".jpg")): 
            img_path = os.path.join(self.img_dir +"_1", img_id + ".jpg")
        else: 
            img_path = os.path.join(self.img_dir +"_2", img_id + ".jpg")

        image = Image.open(img_path)
        label = int(self.metadata.iloc[idx]['dx'])
        if self.transform: 
            image = self.transform(image) 
        if self.target_transform: 
            label = self.target_transform(label)

        return image, label

In [ ]:
image_loader = CustomDataSetLoader(metadata, 'HAM10000_images_part', transform=tranform)

In [ ]:
image1, label1 = image_loader[1]

In [ ]:
image1

In [ ]:
label1

In [ ]:
images, labels = image_loader[0]

In [ ]:
images.shape

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
from sklearn.model_selection import train_test_split

In [84]:
train_df, temp_df = train_test_split(metadata, test_size=0.3, stratify=metadata['dx'], random_state=42)

val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['dx'], random_state=42)

In [85]:
train_df_loader = CustomDataSetLoader(train_df, 'HAM10000_images_part', transform=tranform)

In [86]:
test_df_loader = CustomDataSetLoader(test_df, 'HAM10000_images_part', transform=tranform)

In [87]:
val_df_loader = CustomDataSetLoader(val_df, 'HAM10000_images_part', transform=tranform)

In [88]:
torch.manual_seed(42)
train_loader = DataLoader(train_df_loader, batch_size=32, shuffle=True)
valid_loader = DataLoader(val_df_loader, batch_size=32)
test_loader = DataLoader(test_df_loader, batch_size=32)

In [89]:
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}, Labels shape: {labels.shape}")

Batch shape: torch.Size([32, 3, 224, 224]), Labels shape: torch.Size([32])


In [91]:
import torchvision

In [96]:
weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1

In [100]:
device = 'cuda'

In [101]:
model = torchvision.models.resnet18(weights=weights).to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Tarun V/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|█████████████████████████████████████████████████████████████████████████████| 44.7M/44.7M [00:03<00:00, 14.9MB/s]


In [102]:
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [103]:
transform = weights.transforms()

In [109]:
[name for name, child in model.named_children()]

['conv1',
 'bn1',
 'relu',
 'maxpool',
 'layer1',
 'layer2',
 'layer3',
 'layer4',
 'avgpool',
 'fc']

In [113]:
model.fc

Linear(in_features=512, out_features=1000, bias=True)

In [114]:
for param in model.parameters(): 
    param.requires_grad = False 

for param in model.fc.parameters():
    param.requires_grad = True

In [115]:
import torch.nn as nn

In [120]:
model.fc = nn.Linear(in_features=512, out_features=7, bias=True).to(device)

In [121]:
model.fc

Linear(in_features=512, out_features=7, bias=True)

In [141]:
import torchmetrics

def evaluate_tm(model, data_loader, metric): 
    model.eval()
    metric.reset()
    with torch.no_grad(): 
        for X_batch, y_batch in data_loader: 
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)

    return metric.compute()


def train(model, optimizer, loss_fn, metric, train_loader, valid_loader, n_epochs): 
    history = {"train_losses" : [], "train_metrics" : [], "valid_metrics" : []}
    for epoch in range(n_epochs):
        total_loss = 0
        metric.reset()
        model.train()
        for X_batch, y_batch in train_loader: 
            X_batch, y_batch = X_batch.to(device), y_batch.to(device) 
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(evaluate_tm(model, valid_loader, metric).item())

        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

In [127]:
class_counts = train_df['dx'].value_counts().sort_index().values
class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
class_weights = class_weights / class_weights.sum() * len(class_weights)

print(f"Class weights: {class_weights}")

Class weights: tensor([0.0461, 0.2774, 0.2810, 0.6004, 0.9438, 2.1831, 2.6682])


In [142]:
n_epochs = 20
optimizer = torch.optim.AdamW(model.fc.parameters())
xentropy = nn.CrossEntropyLoss(weight=class_weights.to(device))
accuracy = torchmetrics.Accuracy(task="multiclass",
                                 num_classes=7).to(device)
history = train(model, optimizer, xentropy, accuracy,
                train_loader, valid_loader, n_epochs)

Epoch 1/1, train loss: 0.7055, train metric: 0.7151, valid metric: 0.7190


In [146]:
class_weights

tensor([0.0461, 0.2774, 0.2810, 0.6004, 0.9438, 2.1831, 2.6682])

In [151]:
[name for name, child in model.named_children()]

['conv1',
 'bn1',
 'relu',
 'maxpool',
 'layer1',
 'layer2',
 'layer3',
 'layer4',
 'avgpool',
 'fc']

In [152]:
for param in model.parameters(): 
    param.requires_grad = False 

for param in model.layer4.parameters(): 
    param.requires_grad = True

for param in model.layer3.parameters(): 
    param.requires_grad = True

for param in model.fc.parameters(): 
    param.requires_grad = True

In [154]:
n_epochs = 10
optimizer = torch.optim.AdamW(model.fc.parameters())
xentropy = nn.CrossEntropyLoss(weight=class_weights.to(device))
accuracy = torchmetrics.Accuracy(task="multiclass",
                                 num_classes=7).to(device)
history = train(model, optimizer, xentropy, accuracy,
                train_loader, valid_loader, n_epochs)

Epoch 1/10, train loss: 0.6969, train metric: 0.7066, valid metric: 0.7250
Epoch 2/10, train loss: 0.7243, train metric: 0.7121, valid metric: 0.6897
Epoch 3/10, train loss: 0.6740, train metric: 0.7081, valid metric: 0.6445
Epoch 4/10, train loss: 0.6869, train metric: 0.7094, valid metric: 0.6691
Epoch 5/10, train loss: 0.6770, train metric: 0.7158, valid metric: 0.7244
Epoch 6/10, train loss: 0.6745, train metric: 0.7231, valid metric: 0.6984
Epoch 7/10, train loss: 0.6876, train metric: 0.7114, valid metric: 0.7157
Epoch 8/10, train loss: 0.6585, train metric: 0.7220, valid metric: 0.6744
Epoch 9/10, train loss: 0.6867, train metric: 0.7111, valid metric: 0.6764
Epoch 10/10, train loss: 0.6824, train metric: 0.7154, valid metric: 0.6538
